# 03 — Model Training

**Fish Habitat / PFZ — Problem Statement B**

Three model tiers, and one validation decision that matters more than all of
them put together.

## The validation decision

Occurrence records are strongly **spatially autocorrelated**. A random K-fold
split puts points from the same survey transect on both sides of the split, and
the model scores brilliantly by recognising its neighbours — not by learning
anything transferable.

So we use **spatial block cross-validation**: the region is tiled into 3°
(~330 km) blocks, and whole blocks are held out. Held-out points are then far
from any training point, which is the only way to measure whether the model
generalises to water it has never seen.

**Expect the block-CV number to be substantially lower than a random-split
number would be.** That gap is information, not a defect: it is the difference
between a model that learned ecology and one that memorised a map.

## The three tiers

| Tier | Model | Why it is here |
|---|---|---|
| 1 | **MaxEnt** (L1 logistic regression + hinge/quadratic expansion) | the standard ecological baseline; fisheries scientists trust it, so it is a defensible floor |
| 2 | **Random Forest** | captures interactions the linear tier cannot |
| 2 | **LightGBM** | fast, handles missing values natively, SHAP-explainable |
| — | **Skill-weighted ensemble** | no single method dominates across species and regions |

On MaxEnt: this is genuinely MaxEnt's estimator, not a substitute. Phillips &
Dudík (2008) showed MaxEnt's Gibbs distribution is exactly the solution of
regularised logistic regression on presence-background data; Renner & Warton
(2013) tied both to an inhomogeneous Poisson point process.

In [ ]:
import sys, time, warnings
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]
sys.path.insert(0, str(PROJECT_ROOT))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from marine_ml import config, fusion, viz
from marine_ml.validation import splits, metrics
from fish_habitat_prediction.src import features as feature_lib
from fish_habitat_prediction.src import models as model_lib
from fish_habitat_prediction.src import train as train_lib

viz.use_house_style()

featured = fusion.read_feature_store("fish_habitat_points")
model_columns = feature_lib.feature_columns(featured)
frame = feature_lib.drop_unusable_rows(featured, model_columns)

print(f"{len(frame)} rows · {len(model_columns)} features · "
      f"{int(frame.presence.sum())} presences ({frame.presence.mean():.1%})")

## 1. What spatial block CV actually does

Before running it, look at it. The left panel shows a random split; the right
shows the block split. The difference is why the two produce such different
scores.

In [ ]:
BLOCK_DEGREES = 3.0
N_SPLITS = 5

rng = np.random.default_rng(config.RANDOM_SEED)
random_fold = rng.integers(0, N_SPLITS, len(frame))

block_fold = np.full(len(frame), -1)
for i, split in enumerate(splits.spatial_block_splits(
        frame.latitude, frame.longitude,
        n_splits=N_SPLITS, block_degrees=BLOCK_DEGREES, seed=config.RANDOM_SEED)):
    block_fold[split.test] = i

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=True, sharey=True)
for ax, folds, title, subtitle in [
    (axes[0], random_fold, "Random 5-fold split",
     "Folds are interleaved — every test point has training neighbours metres away."),
    (axes[1], block_fold, "Spatial block 5-fold split",
     "Whole 3° blocks held out — test points are far from anything the model saw."),
]:
    for f in range(N_SPLITS):
        mask = folds == f
        ax.scatter(frame.longitude[mask], frame.latitude[mask], s=7, alpha=0.75,
                   color=viz.CATEGORICAL[f], edgecolor="none", label=f"fold {f + 1}")
    ax.set_aspect("equal")
    ax.grid(False)
    viz.label_axes(ax, title=title, subtitle=subtitle,
                   xlabel="longitude (°E)", ylabel="latitude (°N)")
axes[1].legend(loc="lower left", ncol=2, markerscale=2, fontsize=8)
plt.tight_layout()
plt.show()

## 2. Cross-validation

For each fold and each model tier:

1. Refit the **thermal niche on that fold's training presences only** — fitting
   it once outside the loop would leak held-out temperatures into a training
   feature.
2. Fit the model on the training blocks.
3. Score on the held-out block with AUC, PR-AUC, TSS and the Boyce index.

The Boyce index is the one that matters most for presence-only data: it needs
no true absences, which is exactly the situation OBIS leaves us in.

In [ ]:
fold_list = list(splits.spatial_block_splits(
    frame.latitude, frame.longitude,
    n_splits=N_SPLITS, block_degrees=BLOCK_DEGREES, seed=config.RANDOM_SEED))

rows = []
progress = tqdm(total=len(fold_list) * len(model_lib.MODEL_BUILDERS),
                desc="spatial block CV")

for split in fold_list:
    train_frame = frame.iloc[split.train]
    test_frame = frame.iloc[split.test]

    if train_frame.presence.nunique() < 2 or test_frame.presence.nunique() < 2:
        progress.update(len(model_lib.MODEL_BUILDERS))
        continue

    # Per-fold niche fit — the leakage guard.
    niche = feature_lib.fit_thermal_niche(train_frame)
    train_fold = feature_lib.apply_thermal_niche(train_frame, niche)
    test_fold = feature_lib.apply_thermal_niche(test_frame, niche)

    for name, builder in model_lib.MODEL_BUILDERS.items():
        progress.set_postfix_str(f"{split.name} · {name}")
        model = builder(train_fold, model_columns, config.RANDOM_SEED)
        model.fit(train_fold[model_columns], train_fold.presence)
        scores = model.predict_proba(test_fold[model_columns])[:, 1]

        report = metrics.evaluate_classification(
            test_fold.presence.to_numpy(), scores, name=split.name, compute_boyce=True
        )
        rows.append(report.as_row() | {"model": name})
        progress.update(1)

progress.close()
fold_scores = pd.DataFrame(rows)
print(f"\n{len(fold_scores)} fold × model results")

In [ ]:
cv_summary = (
    fold_scores.groupby("model")[["roc_auc", "pr_auc", "tss", "boyce", "precision", "recall"]]
    .agg(["mean", "std"])
    .round(3)
)
cv_summary

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
model_names = sorted(fold_scores.model.unique())
model_color = {name: viz.CATEGORICAL[i] for i, name in enumerate(model_names)}

for ax, metric, label, floor in [
    (axes[0], "roc_auc", "ROC-AUC", 0.5),
    (axes[1], "pr_auc", "PR-AUC", None),
    (axes[2], "tss", "True Skill Statistic", 0.0),
    (axes[3], "boyce", "Boyce index", 0.0),
]:
    for i, name in enumerate(model_names):
        values = fold_scores.loc[fold_scores.model == name, metric].dropna()
        # Per-fold points plus the mean: shows spread, not just a bar.
        ax.scatter(np.full(len(values), i) + rng.normal(0, 0.05, len(values)),
                   values, s=26, color=model_color[name], alpha=0.55, edgecolor="none")
        ax.plot([i - 0.28, i + 0.28], [values.mean()] * 2,
                color=model_color[name], linewidth=2.5, solid_capstyle="round")
    if floor is not None:
        ax.axhline(floor, color=viz.INK_MUTED, linewidth=1)
        ax.text(len(model_names) - 0.5, floor, " no skill", va="center",
                fontsize=8, color=viz.INK_SECONDARY)
    ax.set_xticks(range(len(model_names)))
    ax.set_xticklabels([n.replace("_", "\n") for n in model_names], fontsize=9)
    ax.set_title(label, loc="left", fontsize=11, color=viz.INK)

fig.suptitle("Spatial block cross-validation — per-fold scores", x=0.06, ha="left",
             fontsize=13, fontweight="semibold", color=viz.INK)
fig.text(0.06, 0.94,
         "Dots are individual folds, bars are means. Fold-to-fold spread is large — "
         "different blocks are genuinely different habitats.",
         fontsize=9, color=viz.INK_SECONDARY)
plt.tight_layout(rect=[0, 0, 1, 0.91])
plt.show()

**Read the spread, not just the means.** Fold-to-fold variation is large because
the blocks are genuinely different environments — a block over the Bay of Bengal
shelf poses a different problem from one over the open Arabian Sea. A model with
a high mean and a huge spread is not obviously better than one slightly lower
and more consistent.

## 3. Ensemble weighting

Weight each tier by its cross-validated TSS. This is the ML analogue of the
ensemble-SDM approach standard in ecological modelling (`biomod2`): no single
method dominates across species and regions, so averaging weighted by measured
skill reduces single-model bias.

TSS is 0 at chance, so it is its own floor — a model no better than chance gets
zero weight rather than dragging the ensemble.

In [ ]:
model_scores = fold_scores.groupby("model").tss.mean().to_dict()
weights = model_lib.EnsembleWeights.from_scores(model_scores, floor=0.0)

weight_table = pd.DataFrame({
    "cv_tss": pd.Series(model_scores),
    "ensemble_weight": pd.Series(weights.weights),
}).round(3).sort_values("ensemble_weight", ascending=False)
weight_table

The weights come out near-equal, which is itself a finding: **no tier dominates.**
That is the situation ensembling is designed for. Had one model been clearly
best, a single model would be the simpler and more defensible choice.

## 4. Final fit on a held-out block

One spatial block is now held out that was **not used for model selection** —
a different seed, so a different partition. The models are refit on everything
else and scored once on it.

In [ ]:
final_split = next(iter(splits.spatial_block_splits(
    frame.latitude, frame.longitude,
    n_splits=N_SPLITS, block_degrees=BLOCK_DEGREES, seed=config.RANDOM_SEED + 1)))

train_frame = frame.iloc[final_split.train]
test_frame = frame.iloc[final_split.test]
print(f"train {len(train_frame)} rows ({int(train_frame.presence.sum())} presences)")
print(f"test  {len(test_frame)} rows ({int(test_frame.presence.sum())} presences)")

fitted, holdout, ensemble_weights = train_lib.fit_final(
    train_frame, test_frame, model_columns, model_scores, config.RANDOM_SEED
)
holdout[["model", "n", "n_positive", "roc_auc", "pr_auc", "tss", "boyce",
         "mess_extrapolating_fraction"]].round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))

comparison = pd.DataFrame({
    "cross-validation": fold_scores.groupby("model").tss.mean(),
    "held-out block": holdout.set_index("model").tss,
}).dropna()

x = np.arange(len(comparison))
width = 0.36
ax.bar(x - width / 2, comparison["cross-validation"], width,
       color=viz.PRESENCE, label="cross-validation (mean)")
ax.bar(x + width / 2, comparison["held-out block"], width,
       color=viz.ACCENT, label="held-out block")

ax.set_xticks(x)
ax.set_xticklabels(comparison.index, fontsize=9)
ax.legend(loc="upper right")
ax.axhline(0, color=viz.INK_MUTED, linewidth=1)
viz.label_axes(ax, title="Cross-validation vs held-out block (TSS)",
               subtitle="The drop is expected and honest — the held-out block is one specific unseen habitat, not an average over five.",
               ylabel="True Skill Statistic")
plt.tight_layout()
plt.show()

**Do not read the drop as failure.** The CV number averages over five different
held-out blocks; the final number is one specific block, which may be harder
than average. Both are honest; the CV mean is the better estimate of typical
performance, and the held-out block is the more conservative one.

## 5. Persist the models

In [ ]:
result = train_lib.TrainingResult(
    fold_scores=fold_scores,
    model_scores=model_scores,
    ensemble_weights=ensemble_weights,
    holdout=holdout,
    fitted=fitted,
    feature_columns=model_columns,
    importances=train_lib.explain(
        fitted["lightgbm"],
        feature_lib.apply_thermal_niche(train_frame, fitted["_thermal_niche"]),
        model_columns,
    ),
)
train_lib.save(result, name="fish_habitat")

print(f"models  → {config.MODELS_DIR / 'fish_habitat.joblib'}")
print(f"reports → {config.REPORTS_DIR}")
for path in sorted(config.REPORTS_DIR.glob("fish_habitat*")):
    print(f"    {path.name}  ({path.stat().st_size / 1024:.1f} KB)")

---

## Summary

- **Spatial block CV, never random K-fold.** With clustered occurrence records
  a random split measures memorisation, not generalisation.
- The thermal niche is **refit inside every fold** — the one leakage path that
  would otherwise be easy to miss.
- **No tier dominates**, so the ensemble weights come out near-equal. That is
  the case ensembling exists for.
- Metrics follow the SDM literature — AUC, TSS and the **Boyce index**, which
  needs no true absences.

Next: **`04_model_evaluation`** — SHAP attribution, response curves, the MESS
extrapolation check, and a predicted suitability map, plus an honest accounting
of what this model can and cannot be trusted to do.